In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# 0. パス設定
# =========================================================
BASE = Path("/Users/Tomo/Desktop/Research/Data")

TOKYO_FILES = {
    "table6": BASE / "Raw" / "Census2020" / "table6_1_household" / "h06_01_13.csv",
    "table7": BASE / "Raw" / "Census2020" / "table7_1_housing" / "h07_01_13.csv",
    "table9": BASE / "Raw" / "Census2020" / "table9_labor_force" / "h09_13.csv",
    "table12": BASE / "Raw" / "Census2020" / "table12_occupation" / "h12_13.csv",
}

OUT_DIR = BASE / "Intermediate" / "Census2020" / "tokyo_check_final_unit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 1. 共通関数
# =========================================================
def read_census_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        dtype=str,
        encoding="cp932",
        header=3
    )
    df.columns = [str(c).strip().replace("\n", "") for c in df.columns]
    return df

def normalize_code(code):
    """
    町丁字コードや秘匿先情報を6桁化。
    '-' は集計単位なし/全域行として NA にする。
    """
    if pd.isna(code):
        return pd.NA

    s = str(code).strip()

    if s in ["", "nan", "None", "<NA>", "-"]:
        return pd.NA

    s = s.replace(".0", "")
    s = "".join([c for c in s if c.isdigit()])

    if s == "":
        return pd.NA

    if len(s) >= 6:
        return s[:6]
    return s.zfill(6)

def normalize_code_list(code_str):
    """
    '006001;006002;0130' -> ['006001', '006002', '000130']
    """
    if pd.isna(code_str):
        return []

    s = str(code_str).strip()
    if s in ["", "nan", "None", "<NA>", "-"]:
        return []

    out = []
    for x in s.split(";"):
        x = x.strip()
        if x == "":
            continue
        out.append(normalize_code(x))
    return [x for x in out if pd.notna(x)]

def clean_symbols(df: pd.DataFrame) -> pd.DataFrame:
    """
    国勢調査記号の処理:
    '-' は 0
    'X' は欠損
    """
    df = df.copy()
    df = df.replace({
        "－": "-",
        "—": "-",
        "―": "-",
        "…": pd.NA,
        "… ": pd.NA,
        "X": pd.NA,
        "x": pd.NA,
    })
    return df

def prepare_common(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # まず記号処理
    df = clean_symbols(df)

    # コード列
    df["市区町村コード"] = (
        df["市区町村コード"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

    df["町丁字コード_str"] = (
        df["町丁字コード"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    df["地域階層レベル"] = (
        df["地域階層レベル"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    df["町丁字コード_norm"] = df["町丁字コード"].apply(normalize_code)
    df["秘匿先情報_norm"] = df["秘匿先情報"].apply(normalize_code)
    df["合算地域_list"] = df["合算地域"].apply(normalize_code_list)

    return df

def make_node(muni_code, town_code_norm):
    if pd.isna(muni_code) or pd.isna(town_code_norm):
        return pd.NA
    return f"{muni_code}_{town_code_norm}"

def level_priority(level):
    """
    粗いレベルを優先: 2 > 3 > 4
    小さい値ほど優先
    """
    order = {"2": 0, "3": 1, "4": 2}
    return order.get(str(level), 99)

# =========================================================
# 2. Union-Find で final_unit_code を作る
# =========================================================
def attach_final_unit_code(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()

    parent = {}

    def find(x):
        parent.setdefault(x, x)
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        if pd.isna(x) or pd.isna(y):
            return
        rx = find(x)
        ry = find(y)
        if rx != ry:
            parent[ry] = rx

    # ノード情報辞書
    node_info = {}

    for _, row in work.iterrows():
        muni = row["市区町村コード"]
        code = row["町丁字コード_norm"]
        node = make_node(muni, code)
        if pd.isna(node):
            continue

        node_info[node] = {
            "市区町村コード": muni,
            "町丁字コード_norm": code,
            "地域階層レベル": str(row["地域階層レベル"]).strip() if pd.notna(row["地域階層レベル"]) else None,
            "秘匿処理": str(row["秘匿処理"]).strip() if pd.notna(row["秘匿処理"]) else None,
        }

    # エッジ作成
    for _, row in work.iterrows():
        muni = row["市区町村コード"]
        code = row["町丁字コード_norm"]
        status = str(row["秘匿処理"]).strip() if pd.notna(row["秘匿処理"]) else ""
        target = row["秘匿先情報_norm"]
        merged_list = row["合算地域_list"]

        node = make_node(muni, code)
        if pd.isna(node):
            continue

        find(node)

        # 秘匿地域: 自分と秘匿先情報を結ぶ
        if status == "秘匿地域" and pd.notna(target):
            union(node, make_node(muni, target))

        # 合算地域あり: 自分と合算地域列の各コードを結ぶ
        if status == "合算地域あり":
            for c in merged_list:
                union(node, make_node(muni, c))

    # グループ作成
    all_nodes = [
        make_node(m, c)
        for m, c in zip(work["市区町村コード"], work["町丁字コード_norm"])
    ]
    all_nodes = [x for x in all_nodes if pd.notna(x)]

    groups = {}
    for node in all_nodes:
        root = find(node)
        groups.setdefault(root, []).append(node)

    # final_unit_code は「合算地域あり」を優先
    root_to_final = {}

    for root, members in groups.items():
        infos = [node_info[m] for m in members if m in node_info]

        agg_infos = [x for x in infos if x["秘匿処理"] == "合算地域あり"]

        if len(agg_infos) == 1:
            final_code = agg_infos[0]["町丁字コード_norm"]
        elif len(agg_infos) >= 2:
            agg_infos = sorted(
                agg_infos,
                key=lambda x: (level_priority(x["地域階層レベル"]), x["町丁字コード_norm"])
            )
            final_code = agg_infos[0]["町丁字コード_norm"]
        else:
            # フォールバック: 最小コード
            town_codes = sorted([x["町丁字コード_norm"] for x in infos if pd.notna(x["町丁字コード_norm"])])
            final_code = town_codes[0]

        root_to_final[root] = final_code

    def assign_final_unit_code(muni, town_code_norm):
        node = make_node(muni, town_code_norm)
        if pd.isna(node):
            return pd.NA
        return root_to_final[find(node)]

    work["final_unit_code"] = [
        assign_final_unit_code(m, c)
        for m, c in zip(work["市区町村コード"], work["町丁字コード_norm"])
    ]
    work["final_key"] = work["市区町村コード"] + "_" + work["final_unit_code"]

    return work

# =========================================================
# 3. keep_finest 作成
# =========================================================
def attach_keep_finest(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()

    def has_finer_descendant(row, code_list):
        code = row["町丁字コード_str"]
        level = str(row["地域階層レベル"]).strip() if pd.notna(row["地域階層レベル"]) else None

        if pd.isna(code) or code in ["", "-"]:
            return False

        # lv4 は最細
        if level == "4":
            return False

        for other in code_list:
            if other == code:
                continue
            if len(str(other)) > len(str(code)) and str(other).startswith(str(code)):
                return True

        return False

    work["has_finer_descendant"] = False

    for muni, g in work.groupby("市区町村コード", sort=False):
        codes_in_muni = g["町丁字コード_str"].dropna().tolist()
        idx = g.index

        work.loc[idx, "has_finer_descendant"] = [
            has_finer_descendant(row, codes_in_muni)
            for _, row in g.iterrows()
        ]

    # 基本は finer descendant がある粗い単位は落とす
    work["keep_finest"] = ~work["has_finer_descendant"]

    # ただし「合算地域あり」は公開単位なので残す
    mask_agg = work["秘匿処理"].astype(str).str.strip() == "合算地域あり"
    work.loc[mask_agg, "keep_finest"] = True

    return work

# =========================================================
# 4. 各表の前処理
# =========================================================
def prepare_table6(path: Path) -> pd.DataFrame:
    df = read_census_csv(path)
    df = prepare_common(df)

    # 年齢総数だけ残す
    if "世帯員の年齢による世帯の種類" in df.columns:
        df = df[df["世帯員の年齢による世帯の種類"] == "総数"].copy()

    df = attach_final_unit_code(df)
    df = attach_keep_finest(df)
    df = df[df["keep_finest"]].copy()

    # "-" は 0、X は欠損のあと、数値列変換
    num_cols = [
        "総数",
        "親族のみの世帯",
        "核家族世帯",
        "うち夫婦のみの世帯",
        "うち夫婦と子供から成る世帯",
        "核家族以外の世帯",
        "非親族を含む世帯",
        "単独世帯",
        "世帯の家族類型「不詳」",
        "（再掲）3世代世帯",
        "（再掲）夫65歳以上，妻60歳以上の夫婦のみの世帯",
        "（再掲）65歳以上の単独世帯",
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].replace("-", 0), errors="coerce")

    return df

def prepare_table7(path: Path) -> pd.DataFrame:
    df = read_census_csv(path)
    df = prepare_common(df)
    df = attach_final_unit_code(df)
    df = attach_keep_finest(df)
    df = df[df["keep_finest"]].copy()

    num_cols = [
        "住宅に住む一般世帯",
        "持ち家",
        "公営・都市再生機構・公社の借家",
        "民営の借家",
        "給与住宅",
        "間借り",
        "住宅以外に住む一般世帯",
        "住居の種類「不詳」",
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].replace("-", 0), errors="coerce")

    return df

def prepare_table9(path: Path) -> pd.DataFrame:
    df = read_census_csv(path)
    df = prepare_common(df)

    if "男女" in df.columns:
        df = df[df["男女"] == "総数"].copy()

    df = attach_final_unit_code(df)
    df = attach_keep_finest(df)
    df = df[df["keep_finest"]].copy()

    num_cols = [
        "総数",
        "労働力人口",
        "非労働力人口",
        "労働力状態「不詳」",
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].replace("-", 0), errors="coerce")

    return df

def prepare_table12(path: Path) -> pd.DataFrame:
    df = read_census_csv(path)
    df = prepare_common(df)

    if "男女" in df.columns:
        df = df[df["男女"] == "総数"].copy()

    df = attach_final_unit_code(df)
    df = attach_keep_finest(df)
    df = df[df["keep_finest"]].copy()

    occ_cols = [
        "0_総数",
        "A_管理的職業従事者",
        "B_専門的・技術的職業従事者",
        "C_事務従事者",
        "D_販売従事者",
        "E_サービス職業従事者",
        "F_保安職業従事者",
        "G_農林漁業従事者",
        "H_生産工程従事者",
        "I_輸送・機械運転従事者",
        "J_建設・採掘従事者",
        "K_運搬・清掃・包装等従事者",
        "L_分類不能の職業",
    ]
    for c in occ_cols:
        if c in df.columns:
            # "-" は 0、"X" は欠損
            df[c] = pd.to_numeric(df[c].replace("-", 0), errors="coerce")

    return df

# =========================================================
# 5. final_unit_code単位で集約
# =========================================================
def aggregate_by_final_key(df: pd.DataFrame, sum_cols: list, table_name: str) -> pd.DataFrame:
    group_cols = ["final_key", "市区町村コード", "final_unit_code"]

    keep_meta = [
        "都道府県名", "市区町村名", "大字・町名", "字・丁目名",
        "地域階層レベル", "秘匿処理", "町丁字コード", "町丁字コード_norm"
    ]
    keep_meta = [c for c in keep_meta if c in df.columns]

    agg_dict = {c: "sum" for c in sum_cols if c in df.columns}
    for c in keep_meta:
        agg_dict[c] = "first"

    out = df.groupby(group_cols, dropna=False, as_index=False).agg(agg_dict)
    out["source_table"] = table_name
    return out

# =========================================================
# 6. 4表を読み込み
# =========================================================
df6 = prepare_table6(TOKYO_FILES["table6"])
df7 = prepare_table7(TOKYO_FILES["table7"])
df9 = prepare_table9(TOKYO_FILES["table9"])
df12 = prepare_table12(TOKYO_FILES["table12"])

print("df6:", df6.shape)
print("df7:", df7.shape)
print("df9:", df9.shape)
print("df12:", df12.shape)

# 中間CSV出力
df6.to_csv(OUT_DIR / "tokyo_table6_prepared.csv", index=False, encoding="utf-8-sig")
df7.to_csv(OUT_DIR / "tokyo_table7_prepared.csv", index=False, encoding="utf-8-sig")
df9.to_csv(OUT_DIR / "tokyo_table9_prepared.csv", index=False, encoding="utf-8-sig")
df12.to_csv(OUT_DIR / "tokyo_table12_prepared.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 7. final_key単位で集約
# =========================================================
df6a = aggregate_by_final_key(df6, [
    "総数",
    "親族のみの世帯",
    "核家族世帯",
    "うち夫婦のみの世帯",
    "うち夫婦と子供から成る世帯",
    "核家族以外の世帯",
    "非親族を含む世帯",
    "単独世帯",
    "世帯の家族類型「不詳」",
    "（再掲）3世代世帯",
    "（再掲）夫65歳以上，妻60歳以上の夫婦のみの世帯",
    "（再掲）65歳以上の単独世帯",
], "table6")

df7a = aggregate_by_final_key(df7, [
    "住宅に住む一般世帯",
    "持ち家",
    "公営・都市再生機構・公社の借家",
    "民営の借家",
    "給与住宅",
    "間借り",
    "住宅以外に住む一般世帯",
    "住居の種類「不詳」",
], "table7")

df9a = aggregate_by_final_key(df9, [
    "総数",
    "労働力人口",
    "非労働力人口",
    "労働力状態「不詳」",
], "table9").rename(columns={"総数": "pop15_total"})

df12a = aggregate_by_final_key(df12, [
    "0_総数",
    "A_管理的職業従事者",
    "B_専門的・技術的職業従事者",
    "C_事務従事者",
    "D_販売従事者",
    "E_サービス職業従事者",
    "F_保安職業従事者",
    "G_農林漁業従事者",
    "H_生産工程従事者",
    "I_輸送・機械運転従事者",
    "J_建設・採掘従事者",
    "K_運搬・清掃・包装等従事者",
    "L_分類不能の職業",
], "table12")

df6a.to_csv(OUT_DIR / "tokyo_table6_aggregated.csv", index=False, encoding="utf-8-sig")
df7a.to_csv(OUT_DIR / "tokyo_table7_aggregated.csv", index=False, encoding="utf-8-sig")
df9a.to_csv(OUT_DIR / "tokyo_table9_aggregated.csv", index=False, encoding="utf-8-sig")
df12a.to_csv(OUT_DIR / "tokyo_table12_aggregated.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 8. 共通 final_key で結合
# =========================================================
keys6 = set(df6a["final_key"])
keys7 = set(df7a["final_key"])
keys9 = set(df9a["final_key"])
keys12 = set(df12a["final_key"])

common_keys = keys6 & keys7 & keys9 & keys12
print("common_keys:", len(common_keys))

df6m = df6a[df6a["final_key"].isin(common_keys)].copy()
df7m = df7a[df7a["final_key"].isin(common_keys)].copy()
df9m = df9a[df9a["final_key"].isin(common_keys)].copy()
df12m = df12a[df12a["final_key"].isin(common_keys)].copy()

merged = df6m.merge(
    df7m[["final_key", "住宅に住む一般世帯", "持ち家", "公営・都市再生機構・公社の借家", "民営の借家", "給与住宅", "間借り", "住宅以外に住む一般世帯", "住居の種類「不詳」"]],
    on="final_key", how="left", validate="1:1"
)

merged = merged.merge(
    df9m[["final_key", "pop15_total", "労働力人口", "非労働力人口", "労働力状態「不詳」"]],
    on="final_key", how="left", validate="1:1"
)

merged = merged.merge(
    df12m[[
        "final_key", "0_総数",
        "A_管理的職業従事者", "B_専門的・技術的職業従事者", "C_事務従事者",
        "D_販売従事者", "E_サービス職業従事者", "F_保安職業従事者",
        "G_農林漁業従事者", "H_生産工程従事者", "I_輸送・機械運転従事者",
        "J_建設・採掘従事者", "K_運搬・清掃・包装等従事者", "L_分類不能の職業"
    ]],
    on="final_key", how="left", validate="1:1"
)

print("merged shape:", merged.shape)

# =========================================================
# 9. ADI構成変数を作る
# =========================================================
merged["old_couple_rate"] = (
    merged["（再掲）夫65歳以上，妻60歳以上の夫婦のみの世帯"] / merged["総数"]
)

merged["old_single_rate"] = (
    merged["（再掲）65歳以上の単独世帯"] / merged["総数"]
)

merged["single_parent_proxy_rate"] = (
    (merged["核家族世帯"]
     - merged["うち夫婦のみの世帯"]
     - merged["うち夫婦と子供から成る世帯"])
    / merged["総数"]
)

merged["rent_rate"] = (
    (merged["住宅に住む一般世帯"] - merged["持ち家"])
    / merged["住宅に住む一般世帯"]
)

merged["sales_service_rate"] = (
    (merged["D_販売従事者"]
     + merged["E_サービス職業従事者"]
     + merged["F_保安職業従事者"])
    / merged["0_総数"]
)

merged["agriculture_rate"] = (
    merged["G_農林漁業従事者"] / merged["0_総数"]
)

merged["blue_collar_rate"] = (
    (merged["H_生産工程従事者"]
     + merged["I_輸送・機械運転従事者"]
     + merged["J_建設・採掘従事者"]
     + merged["K_運搬・清掃・包装等従事者"])
    / merged["0_総数"]
)

merged["unemployment_rate"] = (
    (merged["労働力人口"] - merged["0_総数"]) / merged["労働力人口"]
)

# =========================================================
# 10. ADI計算
# =========================================================
merged["ADI_raw"] = (
    2.99 * merged["old_couple_rate"]
    + 7.57 * merged["old_single_rate"]
    + 17.4 * merged["single_parent_proxy_rate"]
    + 2.22 * merged["rent_rate"]
    + 4.03 * merged["sales_service_rate"]
    + 6.05 * merged["agriculture_rate"]
    + 5.38 * merged["blue_collar_rate"]
    + 18.3 * merged["unemployment_rate"]
)

# =========================================================
# 11. 確認用出力
# =========================================================
print("\nADI missing:", merged["ADI_raw"].isna().sum())
print("ADI matched:", merged["ADI_raw"].notna().sum())

print("\n構成変数欠損数:")
check_cols = [
    "old_couple_rate",
    "old_single_rate",
    "single_parent_proxy_rate",
    "rent_rate",
    "sales_service_rate",
    "agriculture_rate",
    "blue_collar_rate",
    "unemployment_rate",
]
print(merged[check_cols].isna().sum())

merged.to_csv(OUT_DIR / "tokyo_adi_merged_check.csv", index=False, encoding="utf-8-sig")

# グループ構成確認用
group_check = (
    df12.groupby(["市区町村コード", "final_unit_code"], dropna=False)
    .agg(
        n_rows=("町丁字コード", "size"),
        members=("町丁字コード_norm", lambda x: sorted(set([str(v) for v in x if pd.notna(v)]))),
        levels=("地域階層レベル", lambda x: sorted(set([str(v) for v in x if pd.notna(v)]))),
        statuses=("秘匿処理", lambda x: sorted(set([str(v) for v in x if pd.notna(v) and str(v).strip() != ""]))),
        keep_finest_sum=("keep_finest", "sum"),
    )
    .reset_index()
)

group_check.to_csv(OUT_DIR / "tokyo_final_unit_group_check.csv", index=False, encoding="utf-8-sig")

print("\nSaved files:")
for p in sorted(OUT_DIR.glob("*.csv")):
    print(p)

df6: (5666, 32)
df7: (5666, 29)
df9: (5666, 24)
df12: (5666, 33)
common_keys: 5522
merged shape: (5522, 49)

ADI missing: 207
ADI matched: 5315

構成変数欠損数:
old_couple_rate             202
old_single_rate             202
single_parent_proxy_rate    202
rent_rate                   206
sales_service_rate          197
agriculture_rate            197
blue_collar_rate            197
unemployment_rate           197
dtype: int64

Saved files:
/Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/tokyo_check_final_unit/tokyo_adi_merged_check.csv
/Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/tokyo_check_final_unit/tokyo_final_unit_group_check.csv
/Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/tokyo_check_final_unit/tokyo_table12_aggregated.csv
/Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/tokyo_check_final_unit/tokyo_table12_prepared.csv
/Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/tokyo_check_final_unit/tokyo_table6_aggregated.csv
/Users/Tomo/De

In [15]:
# ---------------------------------------------------------
# 1. ADI欠損行を抽出
# ---------------------------------------------------------
adi_missing = merged[merged["ADI_raw"].isna()].copy()

print("ADI missing rows:", len(adi_missing))

show_cols = [
    "final_key",
    "市区町村コード",
    "都道府県名",
    "市区町村名",
    "大字・町名",
    "字・丁目名",
    "総数",
    "住宅に住む一般世帯",
    "労働力人口",
    "0_総数",
    "old_couple_rate",
    "old_single_rate",
    "single_parent_proxy_rate",
    "rent_rate",
    "sales_service_rate",
    "agriculture_rate",
    "blue_collar_rate",
    "unemployment_rate",
]
show_cols = [c for c in show_cols if c in adi_missing.columns]

print(adi_missing[show_cols].head(30))

ADI missing rows: 207
        final_key 市区町村コード 都道府県名 市区町村名   大字・町名 字・丁目名   総数  住宅に住む一般世帯  労働力人口  \
4    13101_000120   13101   東京都  千代田区    皇居外苑  None  0.0        0.0    0.0   
24   13101_000540   13101   東京都  千代田区   神田相生町  None  0.0        0.0    0.0   
30   13101_001002   13101   東京都  千代田区     丸の内   二丁目  0.0        0.0    0.0   
31   13101_001003   13101   東京都  千代田区     丸の内   三丁目  0.0        0.0    0.0   
32   13101_002001   13101   東京都  千代田区     大手町   一丁目  0.0        0.0    0.0   
33   13101_003002   13101   東京都  千代田区     内幸町   二丁目  0.0        0.0    0.0   
35   13101_005001   13101   東京都  千代田区     霞が関   一丁目  0.0        0.0    0.0   
36   13101_005002   13101   東京都  千代田区     霞が関   二丁目  0.0        0.0    0.0   
37   13101_005003   13101   東京都  千代田区     霞が関   三丁目  7.0        0.0    0.0   
60   13101_020001   13101   東京都  千代田区     一ツ橋   一丁目  0.0        0.0    0.0   
109  13102_000090   13102   東京都   中央区   浜離宮庭園  None  0.0        0.0    0.0   
122  13102_000390   13102   東京都   中央区   水面

In [16]:
# ---------------------------------------------------------
# 2. どの構成変数が欠損しているかを列として付与
# ---------------------------------------------------------
rate_cols = [
    "old_couple_rate",
    "old_single_rate",
    "single_parent_proxy_rate",
    "rent_rate",
    "sales_service_rate",
    "agriculture_rate",
    "blue_collar_rate",
    "unemployment_rate",
]

for c in rate_cols:
    adi_missing[f"miss_{c}"] = adi_missing[c].isna()

adi_missing["missing_rate_vars"] = adi_missing[rate_cols].isna().apply(
    lambda row: [col for col, miss in row.items() if miss],
    axis=1
)

adi_missing["n_missing_rate_vars"] = adi_missing[rate_cols].isna().sum(axis=1)

print(
    adi_missing[
        ["final_key", "市区町村名", "大字・町名", "字・丁目名", "n_missing_rate_vars", "missing_rate_vars"]
    ].head(30)
)

        final_key 市区町村名   大字・町名 字・丁目名  n_missing_rate_vars  \
4    13101_000120  千代田区    皇居外苑  None                    8   
24   13101_000540  千代田区   神田相生町  None                    8   
30   13101_001002  千代田区     丸の内   二丁目                    8   
31   13101_001003  千代田区     丸の内   三丁目                    8   
32   13101_002001  千代田区     大手町   一丁目                    8   
33   13101_003002  千代田区     内幸町   二丁目                    8   
35   13101_005001  千代田区     霞が関   一丁目                    8   
36   13101_005002  千代田区     霞が関   二丁目                    8   
37   13101_005003  千代田区     霞が関   三丁目                    5   
60   13101_020001  千代田区     一ツ橋   一丁目                    8   
109  13102_000090   中央区   浜離宮庭園  None                    8   
122  13102_000390   中央区   水面調査区  None                    8   
157  13102_012001   中央区  日本橋本石町   一丁目                    8   
158  13102_012002   中央区  日本橋本石町   二丁目                    8   
207  13103_000310    港区       ‐  None                    4   
335  131

In [17]:
# ---------------------------------------------------------
# 3. 元変数も含めて確認
# ---------------------------------------------------------
base_cols = [
    "final_key",
    "市区町村コード",
    "都道府県名",
    "市区町村名",
    "大字・町名",
    "字・丁目名",

    # 分母・分子候補
    "総数",
    "（再掲）夫65歳以上，妻60歳以上の夫婦のみの世帯",
    "（再掲）65歳以上の単独世帯",
    "核家族世帯",
    "うち夫婦のみの世帯",
    "うち夫婦と子供から成る世帯",

    "住宅に住む一般世帯",
    "持ち家",

    "D_販売従事者",
    "E_サービス職業従事者",
    "F_保安職業従事者",
    "G_農林漁業従事者",
    "H_生産工程従事者",
    "I_輸送・機械運転従事者",
    "J_建設・採掘従事者",
    "K_運搬・清掃・包装等従事者",

    "労働力人口",
    "0_総数",
]
base_cols = [c for c in base_cols if c in adi_missing.columns]

print(adi_missing[base_cols].head(30))

        final_key 市区町村コード 都道府県名 市区町村名   大字・町名 字・丁目名   総数  \
4    13101_000120   13101   東京都  千代田区    皇居外苑  None  0.0   
24   13101_000540   13101   東京都  千代田区   神田相生町  None  0.0   
30   13101_001002   13101   東京都  千代田区     丸の内   二丁目  0.0   
31   13101_001003   13101   東京都  千代田区     丸の内   三丁目  0.0   
32   13101_002001   13101   東京都  千代田区     大手町   一丁目  0.0   
33   13101_003002   13101   東京都  千代田区     内幸町   二丁目  0.0   
35   13101_005001   13101   東京都  千代田区     霞が関   一丁目  0.0   
36   13101_005002   13101   東京都  千代田区     霞が関   二丁目  0.0   
37   13101_005003   13101   東京都  千代田区     霞が関   三丁目  7.0   
60   13101_020001   13101   東京都  千代田区     一ツ橋   一丁目  0.0   
109  13102_000090   13102   東京都   中央区   浜離宮庭園  None  0.0   
122  13102_000390   13102   東京都   中央区   水面調査区  None  0.0   
157  13102_012001   13102   東京都   中央区  日本橋本石町   一丁目  0.0   
158  13102_012002   13102   東京都   中央区  日本橋本石町   二丁目  0.0   
207  13103_000310   13103   東京都    港区       ‐  None  0.0   
335  13104_000140   13104   東京都   新宿区   

In [18]:
# ---------------------------------------------------------
# 4. 欠損パターンの頻度
# ---------------------------------------------------------
pattern_counts = (
    adi_missing["missing_rate_vars"]
    .astype(str)
    .value_counts()
    .reset_index()
)
pattern_counts.columns = ["missing_pattern", "n"]

print(pattern_counts.head(20))

                                     missing_pattern    n
0  ['old_couple_rate', 'old_single_rate', 'single...  195
1  ['old_couple_rate', 'old_single_rate', 'single...    7
2                                      ['rent_rate']    3
3  ['rent_rate', 'sales_service_rate', 'agricultu...    1
4  ['sales_service_rate', 'agriculture_rate', 'bl...    1


In [19]:
# ---------------------------------------------------------
# 5. CSV出力
# ---------------------------------------------------------
out_missing = OUT_DIR / "tokyo_adi_missing_rows_check.csv"
out_pattern = OUT_DIR / "tokyo_adi_missing_pattern_counts.csv"

adi_missing.to_csv(out_missing, index=False, encoding="utf-8-sig")
pattern_counts.to_csv(out_pattern, index=False, encoding="utf-8-sig")

print("saved:", out_missing)
print("saved:", out_pattern)

saved: /Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/tokyo_check_final_unit/tokyo_adi_missing_rows_check.csv
saved: /Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/tokyo_check_final_unit/tokyo_adi_missing_pattern_counts.csv


In [20]:
adi_missing[["総数", "住宅に住む一般世帯", "労働力人口", "0_総数"]].describe()

,総数,住宅に住む一般世帯,労働力人口,0_総数
count,207.000000,207.000000,207.000000,207.000000
mean,1.048309,0.019324,8.318841,8.318841
std,12.218384,0.278019,73.058062,73.058062
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000
max,174.000000,4.000000,963.000000,963.000000


In [21]:
adi_missing[
    ["final_key", "市区町村名", "大字・町名", "字・丁目名",
     "総数", "住宅に住む一般世帯", "労働力人口", "0_総数"]
].head(50)

,final_key,市区町村名,大字・町名,字・丁目名,総数,住宅に住む一般世帯,労働力人口,0_総数
4,13101_000120,千代田区,皇居外苑,None,0.0,0.0,0.0,0.0
24,13101_000540,千代田区,神田相生町,None,0.0,0.0,0.0,0.0
30,13101_001002,千代田区,丸の内,二丁目,0.0,0.0,0.0,0.0
31,13101_001003,千代田区,丸の内,三丁目,0.0,0.0,0.0,0.0
32,13101_002001,千代田区,大手町,一丁目,0.0,0.0,0.0,0.0
33,13101_003002,千代田区,内幸町,二丁目,0.0,0.0,0.0,0.0
35,13101_005001,千代田区,霞が関,一丁目,0.0,0.0,0.0,0.0
36,13101_005002,千代田区,霞が関,二丁目,0.0,0.0,0.0,0.0
37,13101_005003,千代田区,霞が関,三丁目,7.0,0.0,0.0,0.0
60,13101_020001,千代田区,一ツ橋,一丁目,0.0,0.0,0.0,0.0
